# 💰 Sales Performance & Product Analysis
> **Author:** Binh Pham | Maven Fuzzy Factory

Revenue trends, gross profit, product-level performance, cross-sell, and refund analysis.

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings; warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi':120, 'figure.figsize':(12,5),
                     'axes.titlesize':13, 'axes.titleweight':'bold'})

BASE  = '..'
PROC  = f'{BASE}/data/processed'
orders   = pd.read_csv(f'{PROC}/cleaned_orders.csv',      parse_dates=['created_at'])
items    = pd.read_csv(f'{PROC}/cleaned_order_items.csv',  parse_dates=['created_at'])
refunds  = pd.read_csv(f'{PROC}/cleaned_order_item_refunds.csv', parse_dates=['created_at'])
products = pd.read_csv(f'{PROC}/cleaned_products.csv',    parse_dates=['created_at'])
master   = pd.read_csv(f'{PROC}/master_orders.csv',       parse_dates=['created_at'])


## 1. Monthly Revenue & Gross Profit Trend

In [ ]:

monthly = orders.groupby('year_month').agg(
    revenue=('price_usd','sum'), gross_profit=('gross_profit','sum'),
    orders=('order_id','nunique')
).reset_index()

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()
x = range(len(monthly))
ax1.bar(x, monthly['revenue'], alpha=0.7, color='#3498db', label='Revenue')
ax1.bar(x, monthly['gross_profit'], alpha=0.7, color='#27ae60', label='Gross Profit')
ax2.plot(x, monthly['orders'], color='#e74c3c', marker='o', linewidth=2, label='Orders')

ax1.set_xticks(x)
ax1.set_xticklabels(monthly['year_month'], rotation=45, ha='right', fontsize=8)
ax1.set_ylabel('USD ($)')
ax2.set_ylabel('Orders', color='#e74c3c')
ax1.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f'${v:,.0f}'))
ax1.set_title('Monthly Revenue, Gross Profit & Order Volume')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.tight_layout()
plt.savefig(f'{BASE}/notebooks/img_monthly_revenue.png', bbox_inches='tight')
plt.show()


## 2. Product Revenue Share

In [ ]:

prod_rev = items.merge(products[['product_id','product_name']], on='product_id')
prod_agg = prod_rev.groupby('product_name').agg(
    revenue=('price_usd','sum'), units=('order_item_id','count'),
    gross_profit=('gross_profit','sum')
).reset_index().sort_values('revenue', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = sns.color_palette('Set2', 4)
axes[0].pie(prod_agg['revenue'], labels=prod_agg['product_name'],
            autopct='%1.1f%%', colors=colors,
            wedgeprops={'edgecolor':'white','linewidth':2})
axes[0].set_title('Revenue Share by Product')

axes[1].bar(prod_agg['product_name'], prod_agg['gross_profit'], color=colors)
axes[1].set_ylabel('Gross Profit ($)')
axes[1].set_title('Gross Profit by Product')
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda v,_: f'${v:,.0f}'))
for i, row in prod_agg.reset_index().iterrows():
    axes[1].text(i, row['gross_profit'] + 2000, f"${row['gross_profit']:,.0f}",
                 ha='center', fontsize=9, fontweight='bold')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig(f'{BASE}/notebooks/img_product_revenue.png', bbox_inches='tight')
plt.show()


## 3. Monthly Product Sales Trend

In [ ]:

prod_monthly = prod_rev.groupby(['year_month','product_name'])['price_usd'].sum().reset_index()
pivot = prod_monthly.pivot(index='year_month', columns='product_name', values='price_usd').fillna(0)

fig, ax = plt.subplots(figsize=(14, 5))
pivot.plot(ax=ax, marker='o', linewidth=2)
ax.set_title('Monthly Revenue per Product')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v,_: f'${v:,.0f}'))
ax.set_xlabel('Month')
ax.set_ylabel('Revenue')
ax.legend(title='Product', bbox_to_anchor=(1.01, 1))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f'{BASE}/notebooks/img_product_trend.png', bbox_inches='tight')
plt.show()


## 4. Refund Rate by Product

In [ ]:

ref_merged = items.merge(refunds[['order_item_id','refund_amount_usd']], on='order_item_id', how='left')
ref_merged = ref_merged.merge(products[['product_id','product_name']], on='product_id')
ref_agg = ref_merged.groupby('product_name').agg(
    units_sold=('order_item_id','count'),
    refunds=('refund_amount_usd','count'),
    total_refunded=('refund_amount_usd','sum')
).reset_index()
ref_agg['refund_rate'] = (ref_agg['refunds'] / ref_agg['units_sold'] * 100).round(2)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ['#e74c3c' if r > ref_agg['refund_rate'].mean() else '#27ae60' for r in ref_agg['refund_rate']]
axes[0].bar(ref_agg['product_name'], ref_agg['refund_rate'], color=colors)
axes[0].axhline(ref_agg['refund_rate'].mean(), color='gray', linestyle='--', label='Average')
axes[0].set_ylabel('Refund Rate (%)')
axes[0].set_title('Refund Rate by Product')
for i, row in ref_agg.iterrows():
    axes[0].text(i, row['refund_rate'] + 0.1, f"{row['refund_rate']:.1f}%", ha='center', fontweight='bold')
plt.sca(axes[0]); plt.xticks(rotation=15, ha='right')

axes[1].bar(ref_agg['product_name'], ref_agg['total_refunded'], color=colors)
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda v,_: f'${v:,.0f}'))
axes[1].set_title('Total Refund Amount by Product')
plt.sca(axes[1]); plt.xticks(rotation=15, ha='right')

plt.tight_layout()
plt.savefig(f'{BASE}/notebooks/img_refund_by_product.png', bbox_inches='tight')
plt.show()


## 5. Gross Margin % Over Time

In [ ]:

margin_trend = orders.groupby('year_month').agg(
    revenue=('price_usd','sum'), gross_profit=('gross_profit','sum')
).reset_index()
margin_trend['margin_pct'] = (margin_trend['gross_profit'] / margin_trend['revenue'] * 100).round(2)

fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(range(len(margin_trend)), margin_trend['margin_pct'], alpha=0.3, color='#9b59b6')
ax.plot(range(len(margin_trend)), margin_trend['margin_pct'], color='#9b59b6', linewidth=2, marker='o')
ax.axhline(margin_trend['margin_pct'].mean(), color='red', linestyle='--', label=f"Avg {margin_trend['margin_pct'].mean():.1f}%")
ax.set_xticks(range(len(margin_trend)))
ax.set_xticklabels(margin_trend['year_month'], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Gross Margin (%)')
ax.set_title('Monthly Gross Margin %')
ax.legend()
plt.tight_layout()
plt.savefig(f'{BASE}/notebooks/img_margin_trend.png', bbox_inches='tight')
plt.show()
